In [0]:
from pyspark.sql import functions as F

In [0]:
# --- Configuration ---
CATALOG = "iran_israel_capstone_project"
SCHEMA = "bronze"
LANDING_PATH = "abfss://capstonecontainer@iranisrael65.dfs.core.windows.net/landing_zone/events/events (1).csv"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
# --- 1. ADLS Landing Zone to Unity Catalog (Bronze) ---
# Note: For manual CSVs, the "API -> Landing" step is done manually via Azure Portal/Storage Explorer

print(f"Reading events data from: {LANDING_PATH}")
events_data = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(LANDING_PATH)

In [0]:
# Add Audit Columns
events_bronze = events_data.withColumn("ingestion_timestamp", F.current_timestamp()) \
                           .withColumn("source_file", F.lit("events_timeline.csv"))

In [0]:
# Write to Unity Catalog
events_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("events")
print("Successfully written to bronze.events")